# K-Sigma Anomaly Detection

This notebook demonstrates the K-Sigma detection algorithm for identifying
radioactive sources in gamma-ray time series data.

## Algorithm Overview

The K-Sigma detector:
1. **Maintains a rolling background** (e.g., last 60 seconds)
2. **Computes statistics**: mean and standard deviation of background count rate
3. **Compares foreground** to background: metric = (foreground - mean) / std
4. **Declares alarm** if metric > k threshold (e.g., 5σ)
5. **Aggregates alarms** that are close in time (< 2 seconds apart)
6. **Does not update background** during alarm states

## Dataset

Using the TopCoder Urban Data Challenge dataset (mobile NaI detector).

**Note:** You must download the TopCoder dataset separately and set
`TOPCODER_DATA_DIR` below to point to it.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from gammaflow import SpectralTimeSeries
from gammaflow.algorithms import KSigmaDetector
from gammaflow.datasets import TopCoderDataset

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
%matplotlib inline

# Path to the TopCoder dataset on your machine
TOPCODER_DATA_DIR = '/PATH/TO/topcoder'

ds = TopCoderDataset(TOPCODER_DATA_DIR)
print(ds)

## Load a Run with a Source

In [ ]:
# Pick the closest Co-60 run
ak = ds.get_answer_key('training')
co60_runs = ak[ak['SourceID'] == 4].sort_values('Speed/Offset', ascending=False)
run_id = int(co60_runs.iloc[0]['RunID'])

listmode, metadata = ds.load_run(run_id)

print(f"Run {run_id}")
print(f"  Source: {metadata['SourceName']}")
print(f"  Source Time: {metadata['SourceTime']:.1f} s")
print(f"  Speed/Offset: {metadata['Speed/Offset']:.2f}")
print(f"  {listmode}")

## Convert to Time Series

In [ ]:
time_series = SpectralTimeSeries.from_list_mode(
    listmode,
    integration_time=1.0,
    stride_time=1.0,
    energy_bins=512,
    energy_range=(0, 3000),
)

print(f"{time_series}")
print(f"  Time: {time_series.timestamps[0]:.1f} to {time_series.timestamps[-1]:.1f} s")

## Run K-Sigma Detection

In [ ]:
detector = KSigmaDetector(
    k_threshold=5.0,
    background_window=10.0,
    foreground_window=1.0,
    aggregation_gap=2.0,
    min_background_samples=10,
)

metrics = detector.process_time_series(time_series)
summary = detector.get_alarm_summary()

print(f"Alarms: {summary['n_alarms']}")
if summary['n_alarms'] > 0:
    print(f"Peak significance: {summary['max_peak_metric']:.2f} sigma")
    print(f"Total alarm time: {summary['total_alarm_time']:.2f} s")

true_source_time = metadata['SourceTime']
for i, alarm in enumerate(detector.alarms, 1):
    print(f"  {i}. {alarm}")
    if alarm.start_time <= true_source_time <= alarm.end_time:
        print(f"      Captured true source (t={true_source_time:.1f}s)")

## Visualize Results

In [ ]:
times = time_series.timestamps
count_rates = np.array([
    float(s.counts.sum()) / float(
        s.live_time if s.live_time is not None else s.real_time
    )
    for s in time_series.spectra
])

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# Count rate
ax1.step(times, count_rates, where='post', color='black', linewidth=1.5,
         label='Count rate')
ax1.set_ylabel(r'Count Rate (s$^{-1}$)', fontsize=12)
ax1.set_title(
    f'K-Sigma Detection - Run {run_id} ({metadata["SourceName"]})',
    fontsize=14, fontweight='bold',
)
ax1.grid(True, alpha=0.3)
for i, alarm in enumerate(detector.alarms):
    ax1.axvspan(alarm.start_time, alarm.end_time, alpha=0.3, color='red',
                label='Alarm' if i == 0 else '')
if metadata['SourceID'] != 0:
    ax1.axvline(metadata['SourceTime'], color='green', ls='--', lw=2,
                label='True source')
ax1.legend(loc='best')

# K-sigma metric
ax2.step(times, metrics, where='post', color='steelblue', linewidth=1.5,
         label='K-sigma metric')
ax2.axhline(detector.k_threshold, color='red', ls='--', lw=2,
            label=f'Threshold ({detector.k_threshold}\u03c3)')
ax2.axhline(0, color='gray', ls='-', lw=0.5, alpha=0.5)
ax2.set_xlabel('Time (s)', fontsize=12)
ax2.set_ylabel('Significance (\u03c3)', fontsize=12)
ax2.set_title('Alarm Metric', fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.3)
for alarm in detector.alarms:
    ax2.axvspan(alarm.start_time, alarm.end_time, alpha=0.2, color='red')
ax2.legend(loc='best')

plt.tight_layout()
plt.show()

## ROI-Based Detection

Now let's demonstrate improved performance using Energy Regions of Interest (ROIs).

**Strategy**: Instead of using all 512 energy bins, we'll define ROIs that:
1. Capture Co-60's characteristic gamma rays (1173 keV and 1332 keV)
2. Reduce noise from irrelevant energy channels

This should improve signal-to-noise ratio and detection sensitivity.

In [ ]:
from gammaflow.operations.roi import EnergyROI, rebin_time_series_rois, create_roi_collection

# Co-60 emits two gamma rays: 1173 keV and 1332 keV
roi_definitions = [
    (1100, 1250, "Co-60 Peak 1"),
    (1250, 1400, "Co-60 Peak 2"),
]

rois = create_roi_collection(roi_definitions, method="manual")

print("Defined ROIs:")
for roi in rois:
    print(f"  {roi}")

# Rebin time series using ROIs
roi_counts, roi_labels = rebin_time_series_rois(time_series, rois, return_labels=True)

print(f"\nROI-rebinned data shape: {roi_counts.shape}")
print(f"  {time_series.n_spectra} time points x {len(rois)} ROIs")
print(f"  Reduced from 512 bins to {len(rois)} bins ({512 / len(rois):.0f}x)")

# Build a ROI-based SpectralTimeSeries for the K-Sigma detector
roi_edges = np.arange(len(rois) + 1, dtype=float)

live_times_array = np.array([
    s.live_time if s.live_time is not None else s.real_time
    for s in time_series.spectra
])
real_times_array = np.array([s.real_time for s in time_series.spectra])

roi_time_series = SpectralTimeSeries.from_array(
    roi_counts,
    energy_edges=roi_edges,
    timestamps=time_series.timestamps,
    real_times=real_times_array,
    live_times=live_times_array,
)

print(f"\nROI time series: {roi_time_series}")
for i, roi in enumerate(rois):
    print(f"  Bin {i}: {roi.label} = {roi.e_min:.0f}-{roi.e_max:.0f} keV")

## Run K-Sigma on ROI Data

Now we'll run the same K-Sigma detector on the ROI-based time series and compare results.

In [ ]:
detector_roi = KSigmaDetector(
    k_threshold=5.0,
    background_window=10.0,
    foreground_window=1.0,
    aggregation_gap=2.0,
    min_background_samples=10,
)

metrics_roi = detector_roi.process_time_series(roi_time_series)
summary_roi = detector_roi.get_alarm_summary()

print(f"ROI-Based Detection Results")
print(f"{'=' * 60}")
print(f"  Using {len(rois)} ROIs instead of 512 bins")
print(f"  Alarms: {summary_roi['n_alarms']}")
if summary_roi['n_alarms'] > 0:
    print(f"  Peak significance: {summary_roi['max_peak_metric']:.2f} sigma")
    print(f"  Total alarm time: {summary_roi['total_alarm_time']:.2f} s")

true_source_time = metadata['SourceTime']
for i, alarm in enumerate(detector_roi.alarms, 1):
    print(f"  {i}. {alarm}")
    if alarm.start_time <= true_source_time <= alarm.end_time:
        print(f"      Captured true source (t={true_source_time:.1f}s)")

# Comparison
print(f"\n{'=' * 60}")
print(f"Comparison: Full Spectrum vs ROI-Based")
print(f"{'=' * 60}")
print(f"{'Method':<20} {'Alarms':<10} {'Peak sig':<12} {'Alarm time':<12}")
print(f"{'-' * 54}")
print(f"{'Full Spectrum':<20} {summary['n_alarms']:<10} {summary['max_peak_metric']:<12.2f} {summary['total_alarm_time']:<12.2f}")
print(f"{'ROI-Based':<20} {summary_roi['n_alarms']:<10} {summary_roi['max_peak_metric']:<12.2f} {summary_roi['total_alarm_time']:<12.2f}")

if summary['max_peak_metric'] > 0:
    improvement = (summary_roi['max_peak_metric'] - summary['max_peak_metric']) / summary['max_peak_metric'] * 100
    print(f"\nPeak significance change: {improvement:+.1f}%")

## Visualize ROI-Based Results

Compare the full-spectrum and ROI-based detection side-by-side.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10), sharex=True)

roi_count_rates = np.array([
    float(s.counts.sum()) / float(
        s.live_time if s.live_time is not None else s.real_time
    )
    for s in roi_time_series.spectra
])

# --- LEFT COLUMN: Full Spectrum ---
ax1, ax3 = axes[:, 0]

ax1.step(times, count_rates, where='post', color='black', linewidth=1.5,
         label='Count rate')
ax1.set_ylabel(r'Count Rate (s$^{-1}$)', fontsize=12)
ax1.set_title(f'Full Spectrum (512 bins)', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3)
for i, alarm in enumerate(detector.alarms):
    ax1.axvspan(alarm.start_time, alarm.end_time, alpha=0.3, color='red',
                label='Alarm' if i == 0 else '')
    peak_idx = np.argmin(np.abs(times - alarm.peak_time))
    ax1.plot(alarm.peak_time, count_rates[peak_idx],
             'r*', markersize=15, label='Peak' if i == 0 else '')
if metadata['SourceID'] != 0:
    ax1.axvline(metadata['SourceTime'], color='green', ls='--', lw=2,
                label='True source')
ax1.legend(loc='best', fontsize=10)

ax3.step(times, metrics, where='post', color='steelblue', linewidth=1.5,
         label='K-sigma metric')
ax3.axhline(detector.k_threshold, color='red', ls='--', lw=2,
            label=f'Threshold ({detector.k_threshold}\u03c3)')
ax3.axhline(0, color='gray', ls='-', lw=0.5, alpha=0.5)
ax3.set_xlabel('Time (s)', fontsize=12)
ax3.set_ylabel('Significance (\u03c3)', fontsize=12)
ax3.grid(True, alpha=0.3)
for alarm in detector.alarms:
    ax3.axvspan(alarm.start_time, alarm.end_time, alpha=0.2, color='red')
ax3.legend(loc='best', fontsize=10)

# --- RIGHT COLUMN: ROI-Based ---
ax2, ax4 = axes[:, 1]

ax2.step(times, roi_count_rates, where='post', color='black', linewidth=1.5,
         label='Count rate')
ax2.set_ylabel(r'Count Rate (s$^{-1}$)', fontsize=12)
ax2.set_title(f'ROI-Based ({len(rois)} ROIs)', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3)
for i, alarm in enumerate(detector_roi.alarms):
    ax2.axvspan(alarm.start_time, alarm.end_time, alpha=0.3, color='red',
                label='Alarm' if i == 0 else '')
    peak_idx = np.argmin(np.abs(times - alarm.peak_time))
    ax2.plot(alarm.peak_time, roi_count_rates[peak_idx],
             'r*', markersize=15, label='Peak' if i == 0 else '')
if metadata['SourceID'] != 0:
    ax2.axvline(metadata['SourceTime'], color='green', ls='--', lw=2,
                label='True source')
ax2.legend(loc='best', fontsize=10)

ax4.step(times, metrics_roi, where='post', color='steelblue', linewidth=1.5,
         label='K-sigma metric')
ax4.axhline(detector_roi.k_threshold, color='red', ls='--', lw=2,
            label=f'Threshold ({detector_roi.k_threshold}\u03c3)')
ax4.axhline(0, color='gray', ls='-', lw=0.5, alpha=0.5)
ax4.set_xlabel('Time (s)', fontsize=12)
ax4.set_ylabel('Significance (\u03c3)', fontsize=12)
ax4.grid(True, alpha=0.3)
for alarm in detector_roi.alarms:
    ax4.axvspan(alarm.start_time, alarm.end_time, alpha=0.2, color='red')
ax4.legend(loc='best', fontsize=10)

plt.suptitle(
    f'K-Sigma Detection Comparison - Run {run_id} ({metadata["SourceName"]})',
    fontsize=16, fontweight='bold', y=0.995,
)
plt.tight_layout(rect=[0, 0, 1, 0.99])
plt.show()